<a href="https://colab.research.google.com/github/Samridh2001/Collab-Board/blob/main/GPT2_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import GPT2LMHeadModel

In [ ]:
model_hf= GPT2LMHeadModel.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
sd_hf=model_hf.state_dict()

In [ ]:
for k , v in sd_hf.items():
  print(k,v.shape)


transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 

In [ ]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
class CausalSelfAttention(nn.Module):

  def __init__(self, config):

    super().__init__()
    assert config.n_embd % config.n_head

    self.c_attn = nn.Linear(config.n_embd ,3 *config.n_embd)
    self.c_proj = nn.Linear(config.n_embd , config.n_embd)
    self.n_head=config.n_head
    self.n_embd=config.n_embd

    self.register_buffer("bias",torch.tril(torch.ones(config.block_size , config.block_size)).view(1,1,config.block_size , config.block_size))

  def forward(self ,x):
    B,T,C =x.size()

    qkv =self.c_attn(x)
    q ,k ,v=qkv.split(self.n_embd , dim=2)
    k= k.view(B , T ,self.n_head ,C// self.n_head).transpose(1,2)
    q= q.view(B , T ,self.n_head ,C// self.n_head).transpose(1,2)
    v= v.view(B , T ,self.n_head ,C// self.n_head).transpose(1,2)

    att= (q@ k.transpose(-2,-1))*(1.0/math.sqrt(k.size(-1)))
    att = att.masked_fill(self.bias[:,:,:T,:T] == 0 ,float('-inf'))
    att =F.softmax(att ,dim=-1)
    y =att @ v
    y= y.transpose(1,2).contiguous().view(B,T,C)

    y=self.c_proj(y)
    return y

class MLP(nn.Module):

  def __init__(self,config):
    super().__init__()
    self.c_fc = nn.Linear(config.n_embd , 4*config.n_embd)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4*config.n_embd ,  config.n_embd)


class Block(nn.Module):

  def __init__(self,config):
    super.__init__()
    self.ln_1= nn.LayerNorm(config.n_embed)
    self.attn = CausalSelfAttention(config)
    self.ln_2=nn.LayerNorm(config.n_embed)
    self.mlp=MLP(config)

  def forward(self , x):
    x= x+ self.attn(self.ln_1(x))
    x=x +self.mlp(self.ln_2(x))

    return x

@dataclass
class GPTConfig:
  block_size: int =1024
  vocab_size: int =50257
  n_layer : int = 12
  n_head : int =12
  n_embd: int = 768

class GPT(nn.Module):

  def __init__(self,config):
    super().__init__()
    self.config = config

    self.transformer =nn.ModuleDict(dict(
        wte = nn.Embedding(config.vocab_size ,config.n_embd),
        wpe = nn.Embedding(config.block_size,config.n_embd),
        h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        ln_f=nn.LayerNorm(config.n_embd),
    ))

    self.ln_head=nn.Linear(config.n_embd , config.vocab_size ,bias-False)

  @classmethod
  def from_pretrained(cls , model_type):
    from transformers import GPT2LMHeadModel
    print("loading weights from pretrained gpt: %s" % model_type)

    config_args ={
        'gpt2': dict(n_layers=12 ,n_head=12, n_embd=768), #124 M parameters
        'gpt2-medium' : dict(n_layer=24 ,n_head=16 ,n_embd=1024), #350M
        'gpt2-large': dict(n_layer= 36 , n_head=20 , n_embd=1280), # 774M
        'gpt2-xl': dict(n_layer=48 , n_head= 25 , n_embd=1600), #1558M

    }[model_type]
    config_args['vocab_size']=50257
    config_args['block_size']=1024
    config =GPTConfig(**config_args)
    model=GPT(config)
    sd =  model.state_dict()
    sd_keys = sd_keys()
    sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')]

    model_hf =GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = model_hf.state_dict()

    sd_keys_hf= sd_hf.keys()
    sd_keys_hf= [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
    sd_keys_hf = [k for k in sd_keys_hf if not k.endwith('.attn.bias')]
    transposed = ['attn.c_attn.weight','attn.c_proj.weight','mlp.c_fc.weight','mlp.c_proj.weight']

    assert len(sd_keys_hf) == len(sd_keys)
    for k in sd_keys_hf:
      if any(k.endswith(w) for w in transposed):
        assert sd_hf[k].shape[::-1]==sd[k].shape
        with torch.no_grad():
          sd[k].copy_(sd_hf[k].t())
      else:

        assert sd_hf[k].shape==sd[k].shape
        with torch.no_grad():
          sd[k].copy_(sd_hf[k])

    return model



In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    assert config.n_embd % config.n_head == 0
    self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
    self.c_proj = nn.Linear(config.n_embd, config.n_embd)
    self.n_head = config.n_head
    self.n_embd = config.n_embd
    self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))

  def forward(self, x):
    B, T, C = x.size()
    qkv = self.c_attn(x)
    q, k, v = qkv.split(self.n_embd, dim=2)
    k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
    q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
    v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

    att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
    att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
    att = F.softmax(att, dim=-1)
    y = att @ v
    y = y.transpose(1, 2).contiguous().view(B, T, C)
    y = self.c_proj(y)
    return y

class MLP(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

  def forward(self, x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x

class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.attn = CausalSelfAttention(config)
    self.ln_2 = nn.LayerNorm(config.n_embd)
    self.mlp = MLP(config)

  def forward(self, x):
    x = x + self.attn(self.ln_1(x))
    x = x + self.mlp(self.ln_2(x))
    return x

@dataclass
class GPTConfig:
  block_size: int = 1024
  vocab_size: int = 50257
  n_layer: int = 12
  n_head: int = 12
  n_embd: int = 768

class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config
    self.transformer = nn.ModuleDict(dict(
        wte = nn.Embedding(config.vocab_size, config.n_embd),
        wpe = nn.Embedding(config.block_size, config.n_embd),
        h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        ln_f = nn.LayerNorm(config.n_embd),
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

  def forward(self, idx):
    B, T = idx.size()
    assert T <= self.config.block_size

    pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
    pos_emb = self.transformer.wpe(pos)
    tok_emb = self.transformer.wte(idx)
    x = tok_emb + pos_emb

    for block in self.transformer.h:
      x = block(x)

    x = self.transformer.ln_f(x)
    logits = self.lm_head(x)
    return logits

  @classmethod
  def from_pretrained(cls, model_type):
    from transformers import GPT2LMHeadModel
    print("loading weights from pretrained gpt: %s" % model_type)

    config_args = {
        'gpt2': dict(n_layer=12, n_head=12, n_embd=768),
        'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),
        'gpt2-large': dict(n_layer=36, n_head=20, n_embd=1280),
        'gpt2-xl': dict(n_layer=48, n_head=25, n_embd=1600),
    }[model_type]
    config_args['vocab_size'] = 50257
    config_args['block_size'] = 1024
    config = GPTConfig(**config_args)
    model = GPT(config)
    sd = model.state_dict()
    sd_keys = sd.keys()
    sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')]

    model_hf = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = model_hf.state_dict()

    sd_keys_hf = sd_hf.keys()
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

    assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
    for k in sd_keys_hf:
      if any(k.endswith(w) for w in transposed):
        assert sd_hf[k].shape[::-1] == sd[k].shape
        with torch.no_grad():
          sd[k].copy_(sd_hf[k].t())
      else:
        assert sd_hf[k].shape == sd[k].shape
        with torch.no_grad():
          sd[k].copy_(sd_hf[k])

    return model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

num_return_sequences = 5
max_length = 30

model = GPT.from_pretrained('gpt2')
model.eval()
model.to(device)

#prefix tokens
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode("Hello, I am a language model,")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1)
x = tokens.to(device)

torch.manual_seed(42)
if device == 'cuda':
  torch.cuda.manual_seed(42)

while x.size(1) < max_length:
  with torch.no_grad():
    logits = model(x)
    logits = logits[:, -1, :]
    probs = F.softmax(logits, dim=-1)

    topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
    ix = torch.multinomial(topk_probs, 1)
    xcol = torch.gather(topk_indices, -1, ix)
    x = torch.cat((x, xcol), dim=1)

for i in range(num_return_sequences):
  tokens = x[i, :max_length].tolist()
  decoded = enc.decode(tokens)
  print(">", decoded)

Using device: cpu
loading weights from pretrained gpt: gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

> Hello, I am a language model, not a design language.

I have a pretty clear understanding of syntax and grammar.

I try
> Hello, I am a language model, I use an exacted syntax.

"Do you think you can be anything?"

He glanced
> Hello, I am a language model, that's not something to me," she said.

Barker's mother, Julie Barba,
> Hello, I am a language model, not a language program. I am a language creator," she said.

"It is an opportunity for
> Hello, I am a language model, I would like to use it to help my students with any language problems. This tutorial will work for you and
